In [1]:
# General
import os
import numpy as np
import matplotlib.pyplot as plt
import random
# model
import tensorflow
import tensorflow as tf
import pickle
import time
import sys

import keras
from keras.models import Sequential
from keras.layers import Add, GlobalAveragePooling2D,Dense, Flatten, Conv2D, Lambda,Input, BatchNormalization, Activation
from tensorflow.keras.utils import Sequence
from keras.utils import to_categorical
# from keras.callbacks import TensorBoard
from tensorflow.keras.layers import DepthwiseConv2D

from tensorflow.keras.optimizers import AdamW
from tensorflow.keras import *
# from tensorflow.keras import layers as Layers
from tensorflow.keras.layers import *
from keras.activations import relu
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.layers import *
from tensorflow.keras import layers, models

# from tensorflow.keras.layers import Add, GlobalAveragePooling2D,Dense, Flatten, Conv2D, Lambda,	Input, BatchNormalization, Activation
from tensorflow.keras.preprocessing.image import ImageDataGenerator
# from keras.utils import to_categorical
from tensorflow.keras.optimizers import *
from tensorflow.keras.callbacks import *
from tensorflow.keras.callbacks import TensorBoard, ModelCheckpoint,EarlyStopping,ReduceLROnPlateau,LambdaCallback

2024-06-04 06:47:07.782636: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-06-04 06:47:07.782733: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-06-04 06:47:07.908021: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


## Config

In [2]:
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)

In [3]:
sys.path.append('/kaggle/input/resnet-34/')
directory = "/kaggle/working/"
model_dir = "/kaggle/working/"
weight_decay = 1e-4
lr=[5e-3,1e-3,5e-4,1e-4]
num_classes = 10
dropout_rate=0.

BATCH_SIZE = 32
IMG_SIZE = 32
optimizer = AdamW
epoch = 50
loss='categorical_crossentropy'
lr_reducer = ReduceLROnPlateau(factor=np.sqrt(0.1),
                               cooldown=0,
                               patience=5,
                               min_lr=0.5e-6)

# early_stopping = EarlyStopping(monitor='val_accuracy', min_delta=0.001, patience=6, verbose=0, mode='auto', baseline=None, restore_best_weights=True)
# early_stopping = EarlyStopping(monitor='val_loss', min_delta=0.001, patience=5, verbose=0, mode='auto', baseline=None, restore_best_weights=True)

# Data with Normal Augmentation

In [4]:
BATCH_SIZE = 32
IMG_SIZE =32

(X_train, y_train), (X_test, y_test) = cifar10.load_data()

y_train = to_categorical(y_train, num_classes=num_classes)
y_test = to_categorical(y_test, num_classes=num_classes)


X_train = X_train.astype('float32')
X_test = X_test.astype('float32')


X_train /= 255.
X_test /= 255.



steps = int(X_train.shape[0] / BATCH_SIZE)

datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
)
train_generator = datagen.flow(X_train, y_train, batch_size=BATCH_SIZE, shuffle=True)  # generate augmented data from training set

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))

# create the dataset pipeline
train_ds = (train_ds
            .shuffle(1024)
            .cache()
            .repeat()
            .batch(BATCH_SIZE)
            .prefetch(buffer_size=tf.data.experimental.AUTOTUNE))

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


# Model ConvMix

In [5]:
import tensorflow as tf
from tensorflow.keras import layers, models

def activation_block(x, dropout_rate, training):
    x = layers.Activation("gelu")(x)
    x = layers.Dropout(dropout_rate)(x, training=training)
    return layers.BatchNormalization()(x, training=training)

def conv_stem(x, filters: int, patch_size: int, dropout_rate, training):
    x = layers.Conv2D(filters, kernel_size=patch_size, strides=patch_size)(x)
    return activation_block(x, dropout_rate, training)

def conv_mixer_block(x, filters: int, kernel_size: int, dropout_rate, training):
    # Depthwise convolution.
    x0 = x
    x = layers.DepthwiseConv2D(kernel_size=kernel_size, padding="same")(x)
    x = layers.Dropout(dropout_rate)(x, training=training)
    x = layers.Add()([activation_block(x, dropout_rate, training), x0])  # Residual.

    # Pointwise convolution.
    x = layers.Conv2D(filters, kernel_size=1)(x)
    x = layers.Dropout(dropout_rate)(x, training=training)
    x = activation_block(x, dropout_rate, training)

    return x

def get_conv_mixer_512_12(
    image_size=32, filters=512, depth=12, kernel_size=3, patch_size=2, num_classes=num_classes, dropout_rate=dropout_rate
):
    """ConvMixer-256/8: https://openreview.net/pdf?id=TVHS5Y4dNvM.
    The hyperparameter values are taken from the paper.
    """
    inputs = layers.Input((image_size, image_size, 3))
#     x = layers.Rescaling(scale=1.0 / 255)(inputs)

    # Extract patch embeddings.
    x = conv_stem(inputs, filters, patch_size, dropout_rate, training=True)

    # ConvMixer blocks.
    for _ in range(depth):
        x = conv_mixer_block(x, filters, kernel_size, dropout_rate, training=True)

    # Classification block.
    x = layers.GlobalAvgPool2D()(x)
    x = layers.Dropout(dropout_rate)(x, training=True)  # Dropout with rate 0 before final dense layer
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return models.Model(inputs, outputs)

model = get_conv_mixer_512_12(dropout_rate=dropout_rate)

# model.summary()

## Setting up Hyper params

In [6]:
# class GCAdamW(AdamW):
#     def __init__(self, learning_rate, weight_decay, **kwargs):
#         super(GCAdamW, self).__init__(learning_rate, weight_decay, **kwargs)

#     def get_gradients(self, loss, params):
#         # Compute the gradients
#         grads = super().get_gradients(loss, params)

#         # Centralize the gradients
#         centralized_grads = []
#         for grad in grads:
#             if grad is not None:
#                 grad_len = len(grad.shape)
#                 if grad_len > 1:
#                     axis = list(range(grad_len - 1))
#                     grad -= tf.reduce_mean(grad, axis=axis, keepdims=True)
#             centralized_grads.append(grad)
        
#         return centralized_grads
# # optimizer = GCAdamW(learning_rate=0.001, weight_decay=1e-4)

In [7]:

for i in range(len(lr)):
    filepath=f"Bv{lr[i]}.weights.h5"
checkpoint = ModelCheckpoint(filepath=filepath,
                             monitor='val_acc',
                             verbose=1,
                             save_best_only=True,
                            save_weights_only= True)


lr_reducer = ReduceLROnPlateau(factor=np.sqrt(0.1),
                               cooldown=0,
                               patience=5,
                               min_lr=0.5e-6)

callbacks = [checkpoint  ]

### training

In [8]:

model = get_conv_mixer_512_12( dropout_rate=dropout_rate)


filename = f"ConvMixer_10_Normal_{lr[0]:.0e}.weights.h5"


filepath = os.path.join(directory, filename)
checkpoint = ModelCheckpoint(filepath=filepath,
                                      monitor='val_accuracy',
                                      verbose=1,
                                      save_best_only=True,
                                      save_weights_only=True)



optimizer = AdamW(learning_rate=lr[0],weight_decay=weight_decay)


callbacks = [checkpoint, ]


model.compile(
    optimizer=optimizer,
    loss=keras.losses.CategoricalCrossentropy(),
    metrics=['accuracy'],
    )


history =model.fit(train_ds , steps_per_epoch=1360 , epochs=50, validation_data=(X_test, y_test), callbacks=callbacks)


history_save_path = os.path.join(model_dir, f'ConvMixer_10_Normal_{lr[0]:.0e}.pkl')
with open(history_save_path, 'wb') as file_pi:
    pickle.dump(history.history, file_pi)

Epoch 1/50
   2/1360 ━━━━━━━━━━━━━━━━━━━━ 1:08 51ms/step - accuracy: 0.1484 - loss: 2.4683   

I0000 00:00:1717483684.916891     103 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1359/1360 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.3337 - loss: 1.8472
Epoch 1: val_accuracy improved from -inf to 0.26560, saving model to /kaggle/working/ConvMixer_10_Normal_5e-03.weights.h5
1360/1360 ━━━━━━━━━━━━━━━━━━━━ 100s 48ms/step - accuracy: 0.3338 - loss: 1.8468 - val_accuracy: 0.2656 - val_loss: 4.0165
Epoch 2/50
1359/1360 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.5660 - loss: 1.2129
Epoch 2: val_accuracy improved from 0.26560 to 0.55420, saving model to /kaggle/working/ConvMixer_10_Normal_5e-03.weights.h5
1360/1360 ━━━━━━━━━━━━━━━━━━━━ 60s 44ms/step - accuracy: 0.5661 - loss: 1.2128 - val_accuracy: 0.5542 - val_loss: 1.3255
Epoch 3/50
1359/1360 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.6499 - loss: 0.9981
Epoch 3: val_accuracy improved from 0.55420 to 0.59760, saving model to /kaggle/working/ConvMixer_10_Normal_5e-03.weights.h5
1360/1360 ━━━━━━━━━━━━━━━━━━━━ 61s 45ms/step - accuracy: 0.6499 - loss: 0.9980 - val_accuracy: 0.5976 - val_loss: 1.2379
Epoch 4/

In [9]:

model = get_conv_mixer_512_12( dropout_rate=dropout_rate)


filename = f"ConvMixer_10_Normal_{lr[1]:.0e}.weights.h5"


filepath = os.path.join(directory, filename)
checkpoint = ModelCheckpoint(filepath=filepath,
                                      monitor='val_accuracy',
                                      verbose=1,
                                      save_best_only=True,
                                      save_weights_only=True)



optimizer = AdamW(learning_rate=lr[1],weight_decay=weight_decay)


callbacks = [checkpoint, ]


model.compile(
    optimizer=optimizer,
    loss=keras.losses.CategoricalCrossentropy(),
    metrics=['accuracy'],
    )


history =model.fit(train_ds , steps_per_epoch=1360 , epochs=50, validation_data=(X_test, y_test), callbacks=callbacks)


history_save_path = os.path.join(model_dir, f'ConvMixer_10_Normal_{lr[1]:.0e}.pkl')
with open(history_save_path, 'wb') as file_pi:
    pickle.dump(history.history, file_pi)

Epoch 1/50
1359/1360 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.4436 - loss: 1.5317
Epoch 1: val_accuracy improved from -inf to 0.55960, saving model to /kaggle/working/ConvMixer_10_Normal_1e-03.weights.h5
1360/1360 ━━━━━━━━━━━━━━━━━━━━ 95s 47ms/step - accuracy: 0.4437 - loss: 1.5313 - val_accuracy: 0.5596 - val_loss: 1.4598
Epoch 2/50
1359/1360 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.6687 - loss: 0.9364
Epoch 2: val_accuracy improved from 0.55960 to 0.67460, saving model to /kaggle/working/ConvMixer_10_Normal_1e-03.weights.h5
1360/1360 ━━━━━━━━━━━━━━━━━━━━ 61s 45ms/step - accuracy: 0.6688 - loss: 0.9363 - val_accuracy: 0.6746 - val_loss: 0.9615
Epoch 3/50
1359/1360 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.7605 - loss: 0.6937
Epoch 3: val_accuracy improved from 0.67460 to 0.74310, saving model to /kaggle/working/ConvMixer_10_Normal_1e-03.weights.h5
1360/1360 ━━━━━━━━━━━━━━━━━━━━ 62s 45ms/step - accuracy: 0.7605 - loss: 0.6936 - val_accuracy: 0.7431 - val_loss: 0.783

In [10]:

model = get_conv_mixer_512_12( dropout_rate=dropout_rate)


filename = f"ConvMixer_10_Normal_{lr[2]:.0e}.weights.h5"


filepath = os.path.join(directory, filename)
checkpoint = ModelCheckpoint(filepath=filepath,
                                      monitor='val_accuracy',
                                      verbose=1,
                                      save_best_only=True,
                                      save_weights_only=True)



optimizer = AdamW(learning_rate=lr[2],weight_decay=weight_decay)


callbacks = [checkpoint, ]


model.compile(
    optimizer=optimizer,
    loss=keras.losses.CategoricalCrossentropy(),
    metrics=['accuracy'],
    )


history =model.fit(train_ds , steps_per_epoch=1360 , epochs=50, validation_data=(X_test, y_test), callbacks=callbacks)


history_save_path = os.path.join(model_dir, f'ConvMixer_10_Normal_{lr[2]:.0e}.pkl')
with open(history_save_path, 'wb') as file_pi:
    pickle.dump(history.history, file_pi)

Epoch 1/50
1360/1360 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.4687 - loss: 1.4618
Epoch 1: val_accuracy improved from -inf to 0.62810, saving model to /kaggle/working/ConvMixer_10_Normal_5e-04.weights.h5
1360/1360 ━━━━━━━━━━━━━━━━━━━━ 95s 47ms/step - accuracy: 0.4688 - loss: 1.4616 - val_accuracy: 0.6281 - val_loss: 1.2024
Epoch 2/50
1359/1360 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.7053 - loss: 0.8467
Epoch 2: val_accuracy improved from 0.62810 to 0.70490, saving model to /kaggle/working/ConvMixer_10_Normal_5e-04.weights.h5
1360/1360 ━━━━━━━━━━━━━━━━━━━━ 60s 44ms/step - accuracy: 0.7053 - loss: 0.8466 - val_accuracy: 0.7049 - val_loss: 0.9215
Epoch 3/50
1359/1360 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.7899 - loss: 0.6096
Epoch 3: val_accuracy improved from 0.70490 to 0.76440, saving model to /kaggle/working/ConvMixer_10_Normal_5e-04.weights.h5
1360/1360 ━━━━━━━━━━━━━━━━━━━━ 60s 44ms/step - accuracy: 0.7900 - loss: 0.6095 - val_accuracy: 0.7644 - val_loss: 0.727

In [ ]:

model = get_conv_mixer_512_12( dropout_rate=dropout_rate)


filename = f"ConvMixer_10_Normal_{lr[3]:.0e}.weights.h5"


filepath = os.path.join(directory, filename)
checkpoint = ModelCheckpoint(filepath=filepath,
                                      monitor='val_accuracy',
                                      verbose=1,
                                      save_best_only=True,
                                      save_weights_only=True)



optimizer = AdamW(learning_rate=lr[3],weight_decay=weight_decay)


callbacks = [checkpoint, ]


model.compile(
    optimizer=optimizer,
    loss=keras.losses.CategoricalCrossentropy(),
    metrics=['accuracy'],
    )


history =model.fit(train_ds , steps_per_epoch=1360 , epochs=50, validation_data=(X_test, y_test), callbacks=callbacks)


history_save_path = os.path.join(model_dir, f'ConvMixer_10_Normal_{lr[3]:.0e}.pkl')
with open(history_save_path, 'wb') as file_pi:
    pickle.dump(history.history, file_pi)

Epoch 1/50
1360/1360 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.4443 - loss: 1.5298
Epoch 1: val_accuracy improved from -inf to 0.63410, saving model to /kaggle/working/ConvMixer_10_Normal_1e-04.weights.h5
1360/1360 ━━━━━━━━━━━━━━━━━━━━ 95s 47ms/step - accuracy: 0.4443 - loss: 1.5297 - val_accuracy: 0.6341 - val_loss: 1.0364
Epoch 2/50
1359/1360 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.6626 - loss: 0.9594
Epoch 2: val_accuracy improved from 0.63410 to 0.70830, saving model to /kaggle/working/ConvMixer_10_Normal_1e-04.weights.h5
1360/1360 ━━━━━━━━━━━━━━━━━━━━ 61s 45ms/step - accuracy: 0.6627 - loss: 0.9592 - val_accuracy: 0.7083 - val_loss: 0.8429
Epoch 3/50
1359/1360 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.7556 - loss: 0.7063
Epoch 3: val_accuracy improved from 0.70830 to 0.73180, saving model to /kaggle/working/ConvMixer_10_Normal_1e-04.weights.h5
1360/1360 ━━━━━━━━━━━━━━━━━━━━ 60s 44ms/step - accuracy: 0.7556 - loss: 0.7062 - val_accuracy: 0.7318 - val_loss: 0.799